# Product Coding Agent - Batch Mode

Inputs:
1. product batch CSV with `input_id` and `PG_name`
2. scrape artifact root where `input_id` is the folder name
3. canonical PG feature CSV with `PG_name,features,type,allowed_values,description`


In [ ]:
%config Completer.use_jedi = False

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Default LLM transport is scraper-compatible AzureOpenAI SDK.
# Set PCT_LLM_DEPLOYMENT/PCA_LLM_DEPLOYMENT in .env or environment before running.
PROJECT_ROOT


In [ ]:
from product_coding_tool import ProductBatchCodingAgent, ProductBatchCodingRequest
from product_coding_tool.inputs.product_batch import ProductBatchInputProvider
from product_coding_tool.rules.pg_input import PGFeatureInputProvider


## Configure batch inputs

The `input_id` in the product batch CSV must match the folder under `scraped_root`, for example `data/scraped/ROW_0001`.


In [ ]:
batch_input_csv = PROJECT_ROOT / 'examples' / 'product_batch_input_with_pg_name.csv'
scraped_root = PROJECT_ROOT / 'data' / 'scraped'
pg_feature_input_csv = PROJECT_ROOT / 'examples' / 'pg_feature_coding_input.csv'
output_dir = PROJECT_ROOT / 'data' / 'coded' / 'batch_demo'

max_parallel_features = 4
limit_products = 2      # set None for full batch
limit_features = 3      # set None for all PG features
input_ids = None        # example: ['ROW_0001', 'ROW_0002']

batch_input_csv, scraped_root, pg_feature_input_csv, output_dir


In [ ]:
product_provider = ProductBatchInputProvider.from_file(batch_input_csv)
pg_provider = PGFeatureInputProvider.from_file(pg_feature_input_csv)

print('Product rows:', len(product_provider.rows))
print('First 5 input_ids:', product_provider.input_ids()[:5])
print('Available PGs in feature input:', len(pg_provider.pg_names()))
for name in pg_provider.pg_names()[:10]:
    print(' -', name)


## Run batch product coding

For each product row: `input_id → scraped_root/input_id`, `PG_name → PG feature list`, then feature coding runs in parallel workers.


In [ ]:
request = ProductBatchCodingRequest(
    batch_input_csv=batch_input_csv,
    scraped_root=scraped_root,
    pg_feature_input_csv=pg_feature_input_csv,
    output_dir=output_dir,
    input_ids=input_ids,
    limit_products=limit_products,
    limit_features=limit_features,
    max_parallel_features=max_parallel_features,
    llm_preflight=True,
)

result = ProductBatchCodingAgent().run(request)
print('Products coded:', len(result.products))
print('Products failed:', len(result.failed_products))
print('Output dir:', result.output_dir)


In [ ]:
# Compact combined output preview
try:
    import pandas as pd
    combined = output_dir / 'combined_coded_features.csv'
    display(pd.read_csv(combined).head(30))
except Exception as exc:
    print('Preview failed:', exc)
    for product in result.products[:3]:
        print(product.product_id, len(product.results))
